In [1]:
import os
import pandas as pd
import joblib
import ast
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, f1_score, roc_auc_score, recall_score

In [2]:
model = joblib.load(r"C:\Users\oozgen\OneDrive - Nobel\Belgeler\Tez\v1\3. Gereç ve Yöntem\Code_Dataset\5. work model\RF_acc_dpc_bestModel.joblib")

In [3]:
# Veri setini yükle
df = pd.read_csv(r"C:\Users\oozgen\OneDrive - Nobel\Belgeler\Tez\v1\3. Gereç ve Yöntem\Code_Dataset\2. feature selection\merged-ind-features.csv")

# 'sequence' ve 'One_Hot_Encoded' kolonlarını çıkar
df = df.drop(["sequence", "numeric_sequence"], axis=1)

# AAC ve DPC sütunlarını işle
for col in ["AAC", "DPC"]:
    col_df = df[col].apply(lambda x: pd.Series(ast.literal_eval(x)))
    col_df = col_df.rename(columns=lambda x: f"{col}_{str(x)}")
    df = pd.concat([df.drop(col, axis=1), col_df], axis=1)

# Sütun isimlerini string formatına dönüştür ve sayısal veri türlerini onayla
df.columns = [str(col) for col in df.columns]
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Eksik değerleri doldur (NaN değerler için)
df.fillna(0, inplace=True)

# Özellikleri ve hedef değişkeni ayır
X = df.drop("anticancer", axis=1)
y = df["anticancer"]

# Veri setini eğitim ve test setlerine ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Özellik ölçeklendirme
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [4]:
# Test verisi üzerinde tahmin yap ve performans metriklerini hesapla
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
cm = confusion_matrix(y_test, predictions)
sensitivity = recall_score(y_test, predictions)
specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1])
precision = precision_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"Accuracy: {accuracy}")
print(f"Sensitivity: {sensitivity}")
print(f"Specificity: {specificity}")
print(f"Precision: {precision}")
print(f"F1 Score: {f1}")
print(f"ROC AUC Score: {roc_auc}")

Accuracy: 0.7875
Sensitivity: 0.90625
Specificity: 0.7083333333333334
Precision: 0.6744186046511628
F1 Score: 0.7733333333333334
ROC AUC Score: 0.8717447916666666


In [5]:
# 10-fold çapraz doğrulama için StratifiedKFold çalıştırılması...
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Performans metriklerini ve tahminleri saklamak için boş listeler
all_scores = {'accuracy': [], 'precision': [], 'recall': [], 'f1_score': [], 'roc_auc': [], 'sensitivity': [], 'specificity': []}
all_predictions = []
all_true_labels = []

# warm_start False olarak ayarlanır, True için doğrulama metodu uyarı veriyor ve sonuçlar benzer çıkıyor
model.set_params(warm_start=False)

# Performans metriklerini ve tahminleri saklamak için boş listeler ve çapraz doğrulama döngüsü
for train_index, test_index in cv.split(X, y):
    X_train_cv, X_test_cv = X.iloc[train_index], X.iloc[test_index]
    y_train_cv, y_test_cv = y.iloc[train_index], y.iloc[test_index]

    # Modeli çapraz doğrulama setiyle eğit
    model.fit(X_train_cv, y_train_cv)
    predictions = model.predict(X_test_cv)
    probabilities = model.predict_proba(X_test_cv)[:, 1]

    # Tahminleri ve gerçek etiketleri sakla
    all_predictions.extend(predictions)
    all_true_labels.extend(y_test_cv)

    # Performans metriklerini hesapla ve sakla
    accuracy = accuracy_score(y_test_cv, predictions)
    precision = precision_score(y_test_cv, predictions)
    recall = recall_score(y_test_cv, predictions)
    f1 = f1_score(y_test_cv, predictions)
    roc_auc = roc_auc_score(y_test_cv, probabilities)

    cm = confusion_matrix(y_test_cv, predictions)
    sensitivity = recall  # Sensitivity ve recall aynı metriktir
    specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if cm[0, 0] + cm[0, 1] != 0 else 0

    all_scores['accuracy'].append(accuracy)
    all_scores['precision'].append(precision)
    all_scores['recall'].append(recall)
    all_scores['f1_score'].append(f1)
    all_scores['roc_auc'].append(roc_auc)
    all_scores['sensitivity'].append(sensitivity)
    all_scores['specificity'].append(specificity)

# Performans metriklerini ve tahminleri Dataframe'ler haline getir
scores_df = pd.DataFrame(all_scores)
predictions_df = pd.DataFrame({'True Labels': all_true_labels, 'Predictions': all_predictions})

# Dataframe'leri Excel dosyalarına kaydet
excel_path_scores = "model_scores.xlsx"
excel_path_predictions = "model_predictions.xlsx"
scores_df.to_excel(excel_path_scores, index=False)
predictions_df.to_excel(excel_path_predictions, index=False)